In [1]:
# ============================================================
# preprocess_monte_cristo.ipynb
# Requires:
#   - /kaggle/input/books/The-Count-of-Monte-Cristo.txt
#   - /kaggle/input/name-counts/The_Count_of_Monte_Cristo_name_counts.csv
# Produces:
#   - /kaggle/working/monte_cristo_constraints.jsonl
# ============================================================

In [2]:
!pip install -U "transformers>=4.44.0" "accelerate>=0.33.0" --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 161.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 40.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [17]:
import json
import math
import re
from pathlib import Path
from typing import List, Dict
from tqdm.auto import tqdm
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [6]:
# ----------------------------
# Paths and basic config
# ----------------------------

BOOK_NAME = "The Count of Monte Cristo"
BOOK_PATH = Path("/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/The Count of Monte Cristo.txt")
NAME_COUNTS_CSV = Path("/kaggle/input/kdsh26-name-counts-csv/The_Count_of_Monte_Cristo_name_counts.csv")
OUT_PATH = Path("/kaggle/working/monte_cristo_constraints.jsonl")

CHUNK_SIZE = 4000  # characters

In [ ]:
LOG_PATH = Path("/kaggle/working/progress_monte_cristo.log")

def log_progress(msg: str):
    """Print + append progress messages."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

In [7]:
# ----------------------------
# Load name counts and define alias map
# ----------------------------

name_df = pd.read_csv(NAME_COUNTS_CSV)  # columns: name,count

# You can inspect to tune aliases:
# display(name_df.head(30))

# Map surface names to canonical characters.
# Extend this as needed based on name_counts CSV.
ALIAS_MAP = {
    # Main protagonist and aliases
    "Edmond Dantès": "Edmond Dantès / Count of Monte Cristo",
    "Dantès": "Edmond Dantès / Count of Monte Cristo",
    "Edmond": "Edmond Dantès / Count of Monte Cristo",
    "Count of Monte Cristo": "Edmond Dantès / Count of Monte Cristo",
    "Monte Cristo": "Edmond Dantès / Count of Monte Cristo",

    # Villefort family
    "Gérard de Villefort": "Gérard de Villefort",
    "Villefort": "Gérard de Villefort",
    "Noirtier de Villefort": "Noirtier de Villefort",
    "Noirtier": "Noirtier de Villefort",
    "Valentine de Villefort": "Valentine de Villefort",
    "Valentine": "Valentine de Villefort",

    # Antagonists and allies
    "Baron Danglars": "Baron Danglars",
    "Danglars": "Baron Danglars",
    "Fernand de Morcerf": "Fernand de Morcerf",
    "Fernand": "Fernand de Morcerf",
    "Morcerf": "Fernand de Morcerf",
    "Caderousse": "Caderousse",
    "Monsieur Morrel": "Monsieur Morrel",
    "Morrel": "Monsieur Morrel",
    "Beauchamp": "Beauchamp",
    "Debray": "Debray",
    "Andrea Cavalcanti": "Andrea Cavalcanti",
    "Cavalcanti": "Andrea Cavalcanti",
    "Ali": "Ali",
    "Bertuccio": "Bertuccio",
    "Doctor d'Avrigny": "Doctor d'Avrigny",
    "Avrigny": "Doctor d'Avrigny",

    # Younger generation, etc.
    "Albert de Morcerf": "Albert de Morcerf",
    "Albert": "Albert de Morcerf",
    "Maximilian Morrel": "Maximilian Morrel",
    "Maximilian": "Maximilian Morrel",
    "Mercédès": "Mercédès",
    "Mercedes": "Mercédès",  # depending on text variant
}

CANONICAL_CHARS = sorted(set(ALIAS_MAP.values()))

In [8]:
# ----------------------------
# Load book and chunking
# ----------------------------

def load_book(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def chunk_book(text: str, chunk_size: int = CHUNK_SIZE):
    n = math.ceil(len(text) / chunk_size)
    for i in range(n):
        yield i, text[i*chunk_size:(i+1)*chunk_size]

def detect_present_canonicals(chunk_text: str) -> List[str]:
    """
    Return list of canonical characters that appear in this chunk
    based on alias map.
    """
    found = set()
    for surface, canon in ALIAS_MAP.items():
        # simple substring/word-boundary check
        pattern = rf"\b{re.escape(surface)}\b"
        if re.search(pattern, chunk_text):
            found.add(canon)
    return list(found)

In [21]:
# ----------------------------
# LLM setup (8B instruct on H100)
# ----------------------------

MODEL_NAME =  "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" #"deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"  #"meta-llama/Meta-Llama-3-8B-Instruct"  # adjust if needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print("Loaded LLM:", MODEL_NAME)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded LLM: deepseek-ai/DeepSeek-R1-Distill-Llama-8B


In [11]:
# ----------------------------
# Prompts for constraint extraction
# ----------------------------

DIMENSIONS = [
    "birthplace",
    "family_role",
    "political_allegiance",
    "health_state",
    "imprisonment_history",
    "expertise",
    "core_motivation",
]

SYSTEM_PROMPT = f"""
You are extracting factual constraints about ONE character from
Alexandre Dumas' "The Count of Monte Cristo".

Focus on dimensions:
- birthplace
- family_role (parent, grandparent, spouse, etc.)
- political_allegiance (Bonapartist, royalist, neutral)
- health_state (paralyzed, ill, healthy, imprisoned)
- imprisonment_history (where, how long, why)
- expertise (e.g., poisons, law, economics, languages)
- core_motivation (revenge, protection of family, ambition)

Output STRICT JSON with schema:

{{
  "character": str,  # canonical name provided
  "book_name": "The Count of Monte Cristo",
  "constraints": [
    {{
      "dimension": str,    # one of: {", ".join(DIMENSIONS)},
      "value": str,
      "polarity": "positive" or "negative",
      "evidence_text": str,
      "chapter_id": str
    }}
  ]
}}

Include only facts clearly stated or strongly implied in the excerpt.
"""

USER_PROMPT_TEMPLATE = """
Book: The Count of Monte Cristo
Canonical character: {character}
Chunk id: {chapter_id}

Excerpt:
\"\"\"{text_chunk}\"\"\"
"""

In [12]:
# ----------------------------
# LLM calling helpers
# ----------------------------

def generate_json_response(system_prompt: str, user_prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Explicit attention mask to avoid the warning and ensure correct behavior
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=model.device)

    eos_id = model.config.eos_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos_id

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic; no temperature/top_p needed
            pad_token_id=pad_id,
            eos_token_id=eos_id,
        )

    gen_ids = output_ids[0, input_ids.shape[-1]:]
    out_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return out_text


def extract_first_json(text: str) -> Dict:
    match = re.search(r"\{.*\}", text, flags=re.S)
    candidate = match.group(0) if match else text
    candidate = candidate.strip().strip("`")
    try:
        return json.loads(candidate)
    except Exception:
        return {}

def call_llm_extract_constraints(canonical_char: str, chapter_id: int, text_chunk: str) -> Dict:
    user_prompt = USER_PROMPT_TEMPLATE.format(
        character=canonical_char,
        chapter_id=chapter_id,
        text_chunk=text_chunk,
    )
    raw_text = generate_json_response(SYSTEM_PROMPT, user_prompt, max_new_tokens=512)
    obj = extract_first_json(raw_text)

    if not isinstance(obj, dict):
        obj = {}

    obj.setdefault("character", canonical_char)
    obj.setdefault("book_name", BOOK_NAME)
    obj.setdefault("constraints", [])

    if not isinstance(obj["constraints"], list):
        obj["constraints"] = []

    cleaned_constraints = []
    for c in obj["constraints"]:
        if not isinstance(c, dict):
            continue
        dim  = c.get("dimension")
        val  = c.get("value")
        pol  = c.get("polarity", "positive")
        evid = c.get("evidence_text", "")
        chap = c.get("chapter_id", str(chapter_id))

        if not dim or not val:
            continue

        cleaned_constraints.append({
            "dimension": dim,
            "value": val,
            "polarity": pol if pol in ("positive", "negative") else "positive",
            "evidence_text": evid,
            "chapter_id": str(chap),
        })

    obj["constraints"] = cleaned_constraints
    return obj

In [13]:
# ----------------------------
# Merge constraints per character
# ----------------------------

def merge_constraints(book_name: str, character: str, objs: List[Dict]) -> Dict:
    merged = []
    seen = set()
    for obj in objs:
        for c in obj.get("constraints", []):
            key = (c["dimension"], c["value"].strip().lower(), c["polarity"])
            if key in seen:
                continue
            seen.add(key)
            merged.append(c)
    return {
        "book_name": book_name,
        "character": character,
        "constraints": merged,
    }

In [19]:
# ----------------------------
# Main pipeline
# ----------------------------

def main():
    text = load_book(BOOK_PATH)
    char_to_objs = {c: [] for c in CANONICAL_CHARS}

    chunks = list(chunk_book(text))  # materialize to know total length for tqdm
    total_chunks = len(chunks)
    log_progress(f"Starting {BOOK_NAME} preprocessing: {total_chunks} chunks")

    for chunk_idx, (chapter_id, chunk) in enumerate(
        tqdm(chunks, desc="Chunks processed", unit="chunk"),
        start=1
    ):
        present = detect_present_canonicals(chunk)
        if not present:
            # log occasionally for empty chunks so you see progress in saved notebook
            if chunk_idx % 25 == 0:
                log_progress(f"Chunk {chunk_idx}/{total_chunks}: no target characters found")
            continue

        log_progress(
            f"Chunk {chunk_idx}/{total_chunks} (id={chapter_id}): "
            f"{len(present)} characters present: {present}"
        )

        # Inner tqdm is optional; you can keep or drop it
        for ci, canon in enumerate(
            tqdm(
                present,
                desc=f"Chars in chunk {chapter_id}",
                unit="char",
                leave=False
            ),
            start=1
        ):
            try:
                obj = call_llm_extract_constraints(canon, chapter_id, chunk)
                char_to_objs[canon].append(obj)
                log_progress(
                    f"  ↳ Processed character {ci}/{len(present)} in chunk {chunk_idx}: {canon}"
                )
            except Exception as e:
                log_progress(f"  ✗ Error for {canon} chunk {chapter_id}: {e}")

    all_outputs = []
    for canon, objs in char_to_objs.items():
        merged = merge_constraints(BOOK_NAME, canon, objs)
        all_outputs.append(merged)
        log_progress(f"Merged constraints for {canon}: {len(merged['constraints'])} constraints")

    with OUT_PATH.open("w", encoding="utf-8") as f:
        for obj in all_outputs:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    log_progress(f"Saved constraints to {OUT_PATH}")



In [ ]:
if __name__ == "__main__":
    main()

Chunks processed:   0%|          | 0/662 [00:00<?, ?chunk/s]

Chars in chunk 0:   0%|          | 0/9 [00:00<?, ?char/s]

Chars in chunk 1:   0%|          | 0/3 [00:00<?, ?char/s]

Chars in chunk 2:   0%|          | 0/3 [00:00<?, ?char/s]